<a href="https://colab.research.google.com/github/chitrangada-juneja/CSC480_HWS/blob/homeworks/HW3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question 2: Language Identification with a Naive Bayes Classifier

In [2]:
from google.colab import drive
import os
drive.mount('/content/drive')
dataset_path =  '/content/drive/MyDrive/data-2'
os.chdir(dataset_path)

# Verify files

!ls languageID
base_path = '/content/drive/MyDrive/CSC580/languageID/'
filepath = f"{base_path}e0.txt"


Mounted at /content/drive
e0.txt	 e15.txt  e2.txt  e8.txt   j13.txt  j19.txt  j6.txt   s11.txt  s17.txt	s4.txt
e10.txt  e16.txt  e3.txt  e9.txt   j14.txt  j1.txt   j7.txt   s12.txt  s18.txt	s5.txt
e11.txt  e17.txt  e4.txt  j0.txt   j15.txt  j2.txt   j8.txt   s13.txt  s19.txt	s6.txt
e12.txt  e18.txt  e5.txt  j10.txt  j16.txt  j3.txt   j9.txt   s14.txt  s1.txt	s7.txt
e13.txt  e19.txt  e6.txt  j11.txt  j17.txt  j4.txt   s0.txt   s15.txt  s2.txt	s8.txt
e14.txt  e1.txt   e7.txt  j12.txt  j18.txt  j5.txt   s10.txt  s16.txt  s3.txt	s9.txt


In [41]:
import numpy as np
from collections import defaultdict

vocab = [chr(ord('a') + i) for i in range(26)] + [' ']    #creating our vocabulary, defined as a-z + whitespace
char_to_idx = {char: idx for idx, char in enumerate(vocab)} # dict mapping each character to its assigned index

def get_bow(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read().lower()  # Convert to lowercase
    # Count characters (ignore non-vocab)
    counts = np.zeros(27)
    for char in text:
        if char in char_to_idx:
            counts[char_to_idx[char]] += 1
    return counts




part a and b : setting up and calculating probabilities

In [42]:
languages = ['E', 'J', 'S']
train_counts = {lang: np.zeros(27) for lang in languages}

for lang in languages:
    for i in range(10):
        filepath = f"languageID/{lang.lower()}{i}.txt"
        train_counts[lang] += get_bow(filepath)

# here we have train_counts= { 'E': [a:34, b:45, c:67...], 'J': [a:111,....], 'S': [a:90, b:44....]}


# Add-1 smoothing for priors (30 docs +  3 languages)

priors = {lang: (10 + 1) / (30 + 3) for lang in languages}
print("Priors:", priors)

theta = {}
for lang in languages:
    total = np.sum(train_counts[lang]) + 27           #denominator of given formula
    theta[lang] = (train_counts[lang] + 1) / total    # we add one to EACH alphabet's count, and then divide, over all alphabets
    print(f"Theta_{lang}:", theta[lang])
    print("\n")

# here , theta={E: [P('a' | E), P('b'|E),....], J: [P('a'| J), ....], S: [......]}




Priors: {'E': 0.3333333333333333, 'J': 0.3333333333333333, 'S': 0.3333333333333333}
Theta_E: [0.06014789 0.01115806 0.02152383 0.021986   0.10530833 0.0189489
 0.01749637 0.04720718 0.05539416 0.00145253 0.00376337 0.02898455
 0.02053347 0.05790308 0.06443946 0.0167701  0.00059422 0.05380959
 0.06615608 0.08008715 0.02667371 0.00930939 0.01551565 0.00118843
 0.01386505 0.00066024 0.1791232 ]


Theta_J: [1.31676325e-01 1.08915730e-02 5.51560427e-03 1.72449906e-02
 6.01829226e-02 3.90979543e-03 1.40333729e-02 3.17670879e-02
 9.69768903e-02 2.37380437e-03 5.73902115e-02 1.46617329e-03
 3.97961321e-02 5.66920338e-02 9.11121972e-02 9.07631083e-04
 1.39635551e-04 4.27982964e-02 4.21699365e-02 5.69713049e-02
 7.05857711e-02 2.79271102e-04 1.97584305e-02 6.98177756e-05
 1.41730084e-02 7.74977309e-03 1.23368009e-01]


Theta_S: [1.04504282e-01 8.25682420e-03 3.75254175e-02 3.97436687e-02
 1.13746996e-01 8.62653275e-03 7.20931666e-03 4.55973874e-03
 4.98490357e-02 6.65475384e-03 3.08090455e-04 5.

part c, d, e

In [43]:
test_file = "languageID/e10.txt"
test_bow = get_bow(test_file)
print("Bag-of-words for E10.txt:", test_bow)
#we have a test_bow containing the character frequencies of our 27 characters of the alphabet for e10.txt


log_likelihood = {}
for lang in ['E', 'J', 'S']:
    log_likelihood[lang] = np.sum(test_bow * np.log(theta[lang]))
print("Log-Likelihoods:", log_likelihood)

likelihood = {lang: np.exp(log_val) for lang, log_val in log_likelihood.items()}

print("Likelihoods:")
for lang in ['E', 'J', 'S']:
    print(f"p(x|{lang}) = {likelihood[lang]:.6e}")  # Scientific notation for small values

#when you print here, the values are very weird- all zeroes. so we do the logsumexp trick
# to normalize the answer
#by subtracting the maximum in log-domain before exponentiating (from piazza)
#This keeps the exponent values closer to 0, avoiding underflow.

max_log = max(log_likelihood.values())
shifted_logs = {y: ll - max_log for y, ll in log_likelihood.items()}
exp_shifted = {y: np.exp(ll) for y, ll in shifted_logs.items()}

# Step 3: Sum and normalize
sum_exp = sum(exp_shifted.values())
probabilities = {y: exp_val / sum_exp for y, exp_val in exp_shifted.items()}

print("Normalized Probabilities:", probabilities)

log_priors = {lang: np.log(priors[lang]) for lang in ['E', 'J', 'S']}
log_joint = {lang: log_likelihood[lang] + log_priors[lang] for lang in log_likelihood}

# Reusing the code we had above for logsumexp trick

# here, we are calcualting exp(logP(x,language)−max) ; this is the "normalizing" calculation here, for each language
max_log_joint = max(log_joint.values())
shifted_log_joint = {y: ll - max_log_joint for y, ll in log_joint.items()}
exp_shifted_joint = {y: np.exp(ll) for y, ll in shifted_log_joint.items()}
sum_exp_joint = sum(exp_shifted_joint.values())

# Compute posteriors
posterior = {y: exp_val / sum_exp_joint for y, exp_val in exp_shifted_joint.items()}

predicted_label = max(posterior.items(), key=lambda x: x[1])[0]

print(f"\nPredicted label: {max(posterior.items(), key=lambda x: x[1])[0]}")


Bag-of-words for E10.txt: [164.  32.  53.  57. 311.  55.  51. 140. 140.   3.   6.  85.  64. 139.
 182.  53.   3. 141. 186. 225.  65.  31.  47.   4.  38.   2. 498.]
Log-Likelihoods: {'E': np.float64(-7841.786386770359), 'J': np.float64(-8759.518886307715), 'S': np.float64(-8452.383194656028)}
Likelihoods:
p(x|E) = 0.000000e+00
p(x|J) = 0.000000e+00
p(x|S) = 0.000000e+00
Normalized Probabilities: {'E': np.float64(1.0), 'J': np.float64(0.0), 'S': np.float64(6.624844174381455e-266)}

Predicted label: E


part f, g : confusion matrix


In [40]:
# creating a function for our testing set; we reuse it for part g

def evaluate_classifier(theta, priors, vocab, char_to_idx, max_rows=None):
    confusion = np.zeros((3, 3), dtype=int)  # Rows: Predicted, Cols: True
    lang_to_idx = {'E': 0, 'J': 1, 'S': 2}

    for true_lang in ['E', 'J', 'S']:
        for i in range(10, 20):  #10 to 19
            filepath = f"languageID/{true_lang.lower()}{i}.txt"
            test_bow = get_bow_rows(filepath, max_rows=max_rows)

            # Compute log-likelihoods
            log_likelihood = {}
            for lang in ['E', 'J', 'S']:
                log_likelihood[lang] = np.sum(test_bow * np.log(theta[lang]))

            # Predict (argmax of log-likelihood + log-prior)
            predicted_lang = max(log_likelihood.keys(),
                               key=lambda y: log_likelihood[y] + np.log(priors[y]))

            # Update confusion matrix
            confusion[lang_to_idx[predicted_lang], lang_to_idx[true_lang]] += 1

    return confusion

#this is similiar to our previous bag of words function, but here we use rows
#maybe i can use the same function from before because max_rows is a optional param?
#honestly don't know, i dont like python, i just did the same thing
#maxRows basically means how much info we consider from the document
#technically i think maxRows is actually maxLines we read from the doc

def get_bow_rows(filepath, max_rows=None):
    counts = np.zeros(27)
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        if max_rows:
            text = ''.join([next(f) for _ in range(max_rows)]).lower()
        else:
            text = f.read().lower()
    for char in text:
        if char in char_to_idx:
            counts[char_to_idx[char]] += 1
    return counts

#printing out our confusion matrix
# the syntax is such that the column headers are the true languages of the doc
# the row headers are the ones we predicted for the doc

confusion_full = evaluate_classifier(theta, priors, vocab, char_to_idx)
print("Confusion Matrix (Full Documents):")
print("\t\tE\tJ\tS")
for i, lang in enumerate(['E', 'J', 'S']):
    print(f"{lang}\t\t{confusion_full[i][0]}\t{confusion_full[i][1]}\t{confusion_full[i][2]}")

Confusion Matrix (Full Documents):
		E	J	S
E		10	0	0
J		0	10	0
S		0	0	10


In [39]:
from collections import defaultdict

# Initialize character counts per language
char_counts_5rows = {lang: np.zeros(27) for lang in ['E', 'J', 'S']}

# Process training files (only first 5 non-empty rows)


for lang in ['E', 'J', 'S']:
    for i in range(10):  # Files [y]0.txt to [y]9.txt
        filepath = f"languageID/{lang.lower()}{i}.txt"
        char_counts_5rows[lang] += get_bow_rows(filepath, max_rows=5)  # Key change: max_rows=5

theta_5rows = {}
for lang in ['E', 'J', 'S']:
    total_chars = np.sum(char_counts_5rows[lang]) + 27  # 27 characters (a-z + space)
    theta_5rows[lang] = (char_counts_5rows[lang] + 1) / total_chars


priors_5rows = {'E': 11/33, 'J': 11/33, 'S': 11/33}  #just hardcoded because we know we have 10+1 docs and 30+3 total ( with 1-smoothing included)
confusion_5rows = evaluate_classifier( theta_5rows, priors_5rows, vocab, char_to_idx, max_rows=5)

print("Confusion Matrix (First 5 Rows):")
print("\t\tE\tJ\tS")
for i, lang in enumerate(['E', 'J', 'S']):
    print(f"{lang}\t\t{confusion_5rows[i][0]}\t{confusion_5rows[i][1]}\t{confusion_5rows[i][2]}")




Confusion Matrix (First 5 Rows):
		E	J	S
E		10	0	2
J		0	10	0
S		0	0	8
